# RiskLens: Merging the tables and building features

**Goal of this notebook:** put everything together into one table that the model can use. This means joining the bureau and previous application summaries onto the main table, fixing the issues I found in the first notebook, and adding a few ratio features.

**What I'll do**
1. Rebuild the summaries (variables don't carry over between notebooks) and merge them
2. Check that the merge didn't create duplicate rows
3. See how many applicants have no history
4. Clean `DAYS_EMPLOYED`, add `AGE` and the ratio features, fill the counts
5. Check the new features
6. Save the result

## 1. Loading the tables and building the summaries

Same code as in notebooks 02 and 03, copied here so this notebook runs on its own.

In [11]:
import pandas as pd
import numpy as np

path = "../data/raw/home-credit-default-risk/"

# main table
app = pd.read_csv(path + "application_train.csv")

# bureau, one row per applicant
bureau = pd.read_csv(path + "bureau.csv")
bureau["ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    bureau_count=("SK_ID_BUREAU", "count"),
    bureau_active=("ACTIVE", "sum"),
    bureau_credit_sum=("AMT_CREDIT_SUM", "sum"),
    bureau_debt_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_max_overdue=("CREDIT_DAY_OVERDUE", "max")
).reset_index()

# previous applications, one row per applicant
prev = pd.read_csv(path + "previous_application.csv")
prev["REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev_agg = prev.groupby("SK_ID_CURR").agg(
    prev_count=("SK_ID_PREV", "count"),
    prev_refused=("REFUSED", "sum"),
    prev_refused_rate=("REFUSED", "mean"),
    prev_avg_credit=("AMT_CREDIT", "mean")
).reset_index()

print(app.shape, bureau_agg.shape, prev_agg.shape)

(307511, 122) (305811, 6) (338857, 5)


Shapes: main table **(307,511, 122)**, bureau summary **(305,811, 6)**, previous applications summary **(338,857, 5)**.

The two summaries have one row per applicant, so they can be joined to the main table without duplicating anyone.

## 2. Merging

I used `how="left"` so that every applicant in the main table is kept, even if they have no bureau or previous application history.

In [12]:
df = app.merge(bureau_agg, on="SK_ID_CURR", how="left")
df = df.merge(prev_agg, on="SK_ID_CURR", how="left")

# the merge should NOT change the number of rows
print(df.shape)
print(len(df) == len(app))

(307511, 131)
True


The merged table has **307,511 rows and 131 columns**. That's the original 122 columns plus 5 from bureau and 4 from previous applications.

`len(df) == len(app)` gives **True**, so the merge didn't add or remove any rows. I check this every time because a bad join key can quietly duplicate rows and the model would then train on the wrong data.

## 3. Applicants with no history

Before filling anything, I counted how many applicants from the main table don't appear in each summary table.

In [13]:
no_prev = (~app["SK_ID_CURR"].isin(prev_agg["SK_ID_CURR"])).sum()
no_bureau = (~app["SK_ID_CURR"].isin(bureau_agg["SK_ID_CURR"])).sum()
print("no previous applications:", no_prev)
print("no bureau history:", no_bureau)

no previous applications: 16454
no bureau history: 44020


**44,020 applicants (14.3%) have no bureau history** and **16,454 (5.4%) have no previous applications**. For them the summary features are `NaN` after the merge.

Bureau history is missing more often. That makes sense, because it only exists if the person had credit at another bank, while previous applications are earlier loans at Home Credit itself. The bureau number also matches what I found in notebook 02.

## 4. Fixing and adding features

| Feature | Why |
|---|---|
| `AGE` | `DAYS_BIRTH` converted to years (from notebook 01) |
| `DAYS_EMPLOYED` fix | the fake value 365243 replaced with `NaN` (notebook 01) |
| `CREDIT_INCOME_RATIO` | credit amount / income, how big the loan is compared to what the person earns |
| `ANNUITY_INCOME_RATIO` | loan annuity (the regular payment) / income, how heavy the payments are compared to income |

I made the ratios because income and credit amount had almost no correlation with default on their own (-0.004 and -0.030), but how one compares to the other might say more.

In [14]:
# things we found in the first notebook
df["AGE"] = -df["DAYS_BIRTH"] / 365
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# ratio features
df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]

# people with no bureau/previous history have NaN in the counts, 0 makes more sense
count_cols = ["bureau_count", "bureau_active", "prev_count", "prev_refused"]
df[count_cols] = df[count_cols].fillna(0)

I filled `bureau_count`, `bureau_active`, `prev_count` and `prev_refused` with **0**, because no history means zero credits and zero refusals.

I did *not* fill the other features (`bureau_credit_sum`, `prev_refused_rate`, ...), because 0 would be misleading there: a refused rate of 0 means "never refused", not "no information". I left those as `NaN` and will let the model handle them.

## 5. Do the new features help?

In [15]:
print(df[["TARGET", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "prev_refused_rate"]].corr()["TARGET"])

TARGET                  1.000000
CREDIT_INCOME_RATIO    -0.007727
ANNUITY_INCOME_RATIO    0.014265
prev_refused_rate       0.077671
Name: TARGET, dtype: float64


| Feature | Correlation with TARGET |
|---|---|
| CREDIT_INCOME_RATIO | -0.008 |
| ANNUITY_INCOME_RATIO | 0.014 |
| prev_refused_rate | 0.078 |

**The ratio features didn't help, at least not in a linear way.** For comparison, `AMT_CREDIT` on its own was -0.030 and `AMT_INCOME_TOTAL` was -0.004. `CREDIT_INCOME_RATIO` (-0.008) is actually weaker than the raw credit amount, and `ANNUITY_INCOME_RATIO` (0.014) is also very small. So my idea that "loan size compared to income" would say more than each one alone didn't work out.

Two things I noticed:
- `ANNUITY_INCOME_RATIO` is positive, which makes sense: the bigger the payment compared to income, the more people default. But the effect is tiny.
- `CREDIT_INCOME_RATIO` is slightly negative, so people with a bigger loan compared to their income default a bit *less*. That's the opposite of what I expected, but at -0.008 it's basically zero, so I wouldn't read anything into it.

`prev_refused_rate` is **0.078**, exactly the same as in notebook 03, so the merge kept that feature intact. It's still the strongest feature I built.

I could be underrating the ratios. Pearson correlation is sensitive to extreme values and some incomes are very high, which can flatten it. Also, correlation only shows linear relations, so a tree model could still find something in these features. I'll keep them for now and look at them again with SHAP once the model is trained.

## 6. Saving the result

I saved the merged table to `data/processed/train_merged.csv` so the modeling notebook can start from here without redoing all the joins. This folder is in `.gitignore`, so the file stays on my computer and doesn't go to GitHub.

In [16]:
df.to_csv("../data/processed/train_merged.csv", index=False)
print(df.shape)

(307511, 134)


## Summary

**What I did**
- Merged the bureau and previous application summaries onto the main table (no rows lost or duplicated)
- Fixed `DAYS_EMPLOYED`, added `AGE` and two ratio features
- Filled the count features with 0 for people with no history (14.3% have no bureau history, 5.4% no previous applications)
- Saved the final table: **307,511 rows x 134 columns**

**What I found**
- `prev_refused_rate` (0.078) is still the strongest feature I created
- The ratio features are weak in correlation (-0.008 and 0.014)

**To check later**
- Which of the new features the model actually uses (SHAP)

**Next:** `05_baseline.ipynb`, split train/validation and fit a logistic regression as the baseline